#### **Health Connect Model** 
HealthConnect Clinic is facing high rates of missed appointments. The goal is to design a machine learning system that predicts whether a patient will attend or miss their scheduled appointment. This will enable proactive interventions such as reminders, rescheduling, or targeted support.  

We will begin by installing the necessary dependencies in our environment:

In [ ]:
%pip install numpy pandas scikit-learn matplotlib seaborn xgboost pyyaml pyarrow imbalanced-learn xgboost

  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 318.8 MB/s eta 0:00:01
    --------------------------------------- 0.5/36.6 MB 318.8 MB/s eta 0:00:01
   - -------------------------------------- 1.0/36.6 MB 1.6 MB/s eta 0:00:23
   - -------------------------------------- 1.6/36.6 MB 2.1 MB/s eta 0:00:17
   -- ------------------------------------- 2.1/36.6 MB 2.1 MB/s eta 0:00:17
   -- ------------------------------------- 2.6/36.6 MB 2.3 MB/s eta 0:00:16
   ---


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


It is also good practice to verify installation:

In [7]:
import numpy as np
import pandas as pd
import sklearn
import xgboost
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print("All libraries loaded successfully!")

All libraries loaded successfully!


Now, unlike our previous project, where we wrote and defined all our functions within our notebook here step by step, we have chosen to adopt a more standard and structured way of processing, testing, and development.

We have built our entire pipeline within smaller components that we can easily call here and run, for cleanliness, and reusability. 

We have predefined a preprocessing.py components that contains all the functions to carry out our data preprocessing, a train.py that does the training on our selected models, and a an evaluate.py that carries out evaluation on each of the models and generates results for analysis and report.

Another reason for doing this is to have a well defined and arranged repo structure that can be easily maneuvered by visitors interested in exploring our models.

So naturally, the first step we will carry out is data preprocessing:

In [7]:
from src.preprocessing import preprocess_pipeline

df = preprocess_pipeline("data/HealthConnect_Appointment_Data.csv")

#### Preprocessing Summary
**Built a robust preprocessing pipeline (preprocessing.py) that:**  

- Handles missing values with SimpleImputer (categorical → most frequent, numeric → median).

- Converts date columns into engineered features (lead_time_days, appointment_dayofweek, appointment_month).

- Drops raw date columns after feature engineering.

- Saves both the processed dataset and the pipeline object for reuse.

**Fixed earlier errors:**

- FileNotFoundError when saving pipeline → solved by creating the models/ directory with os.makedirs(..., exist_ok=True).

Our next step is training:

In [14]:
!python src/train.py

=== Logistic Regression Results ===
Accuracy: 0.527
Precision: 0.5217519872896806
Recall: 0.527
F1 Score: 0.524262848154458

Classification Report:
               precision    recall  f1-score   support

    Attended       0.54      0.55      0.54       463
   Cancelled       0.10      0.08      0.09        52
     No-Show       0.55      0.55      0.55       485

    accuracy                           0.53      1000
   macro avg       0.40      0.39      0.39      1000
weighted avg       0.52      0.53      0.52      1000


=== Random Forest Results ===
Accuracy: 0.576
Precision: 0.5460031064354455
Recall: 0.576
F1 Score: 0.5605973518302285

Classification Report:
               precision    recall  f1-score   support

    Attended       0.57      0.59      0.58       463
   Cancelled       0.00      0.00      0.00        52
     No-Show       0.59      0.62      0.60       485

    accuracy                           0.58      1000
   macro avg       0.38      0.40      0.39      1000

#### Training Summary
**Updated train.py to:**

- Load the processed dataset instead of raw data (avoids missing engineered features like lead_time_days).

- Use stratified train/test split to ensure all classes are represented.

- Train and save both Logistic Regression and Random Forest models.

**Fixed earlier errors:**

- NaN values in Logistic Regression → solved by adding imputers in preprocessing.

- Multiclass metrics error (average='binary') → solved by using average="weighted" for precision, recall, and F1.

- UndefinedMetricWarning (precision ill‑defined for classes with no predictions) → solved by adding zero_division=0 to metric calls.

Last in this workflow, we run evaluate.py to load the saved models and carry out evaluation on them:

In [17]:
!python src/evaluate.py


=== Logistic Regression Evaluation ===
Accuracy: 0.527
Precision: 0.5217519872896806
Recall: 0.527
F1 Score: 0.524262848154458

Classification Report:
               precision    recall  f1-score   support

    Attended       0.54      0.55      0.54       463
   Cancelled       0.10      0.08      0.09        52
     No-Show       0.55      0.55      0.55       485

    accuracy                           0.53      1000
   macro avg       0.40      0.39      0.39      1000
weighted avg       0.52      0.53      0.52      1000


=== Random Forest Evaluation ===
Accuracy: 0.576
Precision: 0.5460031064354455
Recall: 0.576
F1 Score: 0.5605973518302285

Classification Report:
               precision    recall  f1-score   support

    Attended       0.57      0.59      0.58       463
   Cancelled       0.00      0.00      0.00        52
     No-Show       0.59      0.62      0.60       485

    accuracy                           0.58      1000
   macro avg       0.38      0.40      0.39   

#### Evaluation Summary

**Created evaluate.py to:**

- Reload saved models (logistic_regression.pkl, random_forest.pkl).

- Evaluate them on the test set using consistent metrics.

- Print accuracy, precision, recall, F1, and classification report with multiclass support.

**Generate confusion matrices for each model:**

- Saved automatically as PNG files for reproducibility.

- Displayed inline when running in a Jupyter Notebook for immediate visual inspection.

**Fixed earlier overlap with train.py by clarifying purpose:**

- train.py → fits and saves models.

- evaluate.py → reloads and evaluates models only.

#### Key Errors Encountered & Fixes
- FileNotFoundError when saving pipeline → fixed with os.makedirs.

- NaN values in Logistic Regression → fixed with imputers.

- Missing engineered columns (lead_time_days) → fixed by training on processed dataset.

- Multiclass metrics error → fixed with average="weighted".

- UndefinedMetricWarning → fixed with zero_division=0 and stratified splitting.

#### **Week 6 - Integration and Validation**
As a continuation to the development of our model, we would like to improve the quality of our outputs and validate whether those outputs are suitable for the next stage of the project. To do this, we will be integrating the outputs from a Data Scientist into our own already existing outputs as a way of validating. So basically, integration and validation.

Specifically, changes and modificatiosn will be made to the development pipeline within our preprocessing, training and evaluation components. We will then carry out these processes to output more efficient models for subsequent testing and ultimate deployment.

On analysis and comparison of the data from our Data scientist, who will hereafter be referred to as Mercy, we came up with the following deductions:

**1. Clear Overlaps**
- Problem Definition: 
>1. We both framed the task as predicting appointment no‑shows (binary classification).

>2. I excluded “Cancelled” outcomes in our pipeline; Mercy also recommended excluding them for clarity. → Alignment achieved.

- Feature Engineering:

>1. I engineered lead time and appointment day features.

>2. Mercy engineered no_show_rate, is_weekend, and is_urgent.

>3. Together, these features form a richer set for model training. Our pipeline must ensure these engineered features are preserved and passed correctly to the model.

- Baseline Models:

>1. I implemented Logistic Regression and Random Forest baselines.

>2. Mercy started with Logistic Regression and recommended moving toward tree‑based models (Random Forest, XGBoost). → My next step (adding XGBoost) directly complements her recommendation.

- Evaluation Metrics:

>1. We both of emphasized recall, precision, F1, and ROC‑AUC.

>2. This consistency ensures smooth integration: my pipeline can log and validate the same metrics Mercy used.  

**2. Integration Points for Week 6**
- Data Flow:

>1. Mercy’s cleaned dataset (Cleaned_Data.csv) is already encoded and includes engineered features.

>2. Our pipeline should ingest this dataset directly, avoiding duplicate preprocessing.

- Model Integration:

>1. Mercy’s candidate Logistic Regression baseline and her recommendation for Random Forest/XGBoost provide the models I’ll integrate.

>2. Our pipeline structure (/src/train.py, /src/evaluate.py) is ready to wrap her models and run them end‑to‑end.

- Error Analysis

>1. Mercy identified weaknesses (ROC‑AUC ~0.67, recall vs precision trade‑off).

>2. We can embed her error analysis into our pipeline validation checks, logging false positives/negatives and confusion matrices.

- Cross‑Track Dependency

>1. I receive: Cleaned dataset + engineered features + candidate model(s).

>2. I provide: Integrated ML pipeline that runs her model(s), validates outputs, and prepares for Week 7 testing.

**4. Challenges to Watch**
- Feature Leakage: Mercy excluded reminders and waiting time to avoid leakage. Our pipeline must respect this decision.

- Encoding Consistency: Ensure our preprocessing doesn’t overwrite Mercy’s engineered features.

- Class Imbalance: Both noted imbalance issues — our pipeline should integrate SMOTE or class weights.

- Compute Dependency: Random Forest/XGBoost training time may be longer; pipeline logging should capture runtime and resource usage.

So we'll begin by stating the exact changes to be made within our individual components, after which we will re-run the components to reflect the effected changes.

**For preprocessing.py:**
1. Skip duplicate feature engineering

>- Mercy’s Cleaned_Data.csv already includes engineered features (no_show_rate, is_weekend, is_urgent) and one‑hot encoded categorical variables.

>- Update our preprocessing to detect if these columns exist and not re‑encode them.

In [ ]:
# If dataset already has engineered features, skip re-creation
if "no_show_rate" in df.columns:
    print("Using engineered features from Data Science track")
else:
    # create them here if missing

2. Respect exclusions

>- Remove reminder features and waiting_time_minutes if they appear, to avoid leakage.

In [ ]:
leakage_cols = ["reminder_sent", "reminder_channel", "waiting_time_minutes"]
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])

3. Simplify preprocessing

>- Since Mercy’s dataset is already encoded, you can skip the OneHotEncoder block.

>- Keep imputers and scalers for numeric columns only.

**For train.py:**
1. Add XGBoost model (Mercy recommended this).

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=config["random_state"]
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("\n=== XGBoost Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, average="weighted", zero_division=0))
print("Recall:", recall_score(y_test, y_pred_xgb, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred_xgb, average="weighted", zero_division=0))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb, zero_division=0))

joblib.dump(xgb_model, os.path.join(models_dir, "xgboost.pkl"))


2. Handle class imbalance

>- Add SMOTE or class weights before training.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=config["random_state"])
X_train, y_train = smote.fit_resample(X_train, y_train)

3. Feature importance logging

>- For Random Forest and XGBoost, log feature importances to align with Mercy’s feature engineering.

In [ ]:
importances = rf_model.feature_importances_
print("Top RF features:", importances[:10])

**For evaluate.py:**
1. Load XGBoost model

In [ ]:
xgb_model = joblib.load(os.path.join(models_dir, "xgboost.pkl"))
evaluate_model(xgb_model, X_test, y_test, "XGBoost")

2. Add ROC‑AUC metric (Mercy used this).

In [ ]:
from sklearn.metrics import roc_auc_score

print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))

3. Error analysis logging

>- Add counts of false positives and false negatives.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"False Positives: {fp}, False Negatives: {fn}")

Now that we have clearly outlined our effected changes and modifications, we can now re-run the components one after the other.  

For preprocesing.py:   

In [4]:
!python src/preprocessing.py

For train.py:

In [9]:
!python src/train.py

Models trained and saved successfully.


For evaluate.py:

In [10]:
!python src/evaluate.py


=== Logistic Regression Evaluation ===
Accuracy: 0.6160337552742616
Precision: 0.6162088128547089
Recall: 0.6160337552742616
F1 Score: 0.61478699453383

Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.56      0.59       463
           1       0.61      0.67      0.64       485

    accuracy                           0.62       948
   macro avg       0.62      0.61      0.61       948
weighted avg       0.62      0.62      0.61       948

ROC-AUC: 0.6664736924138852
False Positives: 204, False Negatives: 160

=== Random Forest Evaluation ===
Accuracy: 0.6002109704641351
Precision: 0.6000314337518853
Recall: 0.6002109704641351
F1 Score: 0.6000082871607498

Classification Report:
               precision    recall  f1-score   support

           0       0.59      0.58      0.58       463
           1       0.61      0.62      0.61       485

    accuracy                           0.60       948
   macro avg       0.60      0.6